In [2]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# **Final Project: Translation Detection**

**Motivation**: Knowing that a text is a translation is important since the author’s original intentions can become lost in translation. As a reader, if you know that a text is originally written in the language or if it is the result of a translation can increase your awareness and change your interpretation of the text.
Different languages carry many subtle differences that can be hard for humans to perceive, so translations between languages may carry some traces of these language differences and carry patterns that can be detected by machine learning techniques.

**Research Question**: Can NLP algorithms accurately classify whether a text was originally written in English or translated into English from a different language?

In [2]:
meta = pd.read_csv("Complete_Data/TransComp_metadata.csv")
original_eng = pd.read_csv("Complete_Data/Original_samples.csv")
translated = pd.read_csv("Complete_Data/Translation_samples.csv")

#The total num of data (English & Non-English docs) available for us to test
eng_count = (meta['final_lang'] == 'eng').sum()
non_eng_count = (meta['final_lang'] != 'eng').sum()

print("Number of English docs:", eng_count)
print("Number of Non-English docs", non_eng_count)

#The number of features for a document
features_count_eng = (original_eng['docid'] == 'mdp.39015020852250').sum()
features_count_non_eng = (translated['docid'] == 'coo.31924006114809').sum()

print("Number of features for for English doc 'mdp.39015020852250':", features_count_eng)
print("Number of features for for non-English doc 'coo.31924006114809':", features_count_non_eng)


Number of English docs: 10683
Number of Non-English docs 10630
Number of features for for English doc 'mdp.39015020852250': 2668
Number of features for for non-English doc 'coo.31924006114809': 2358


Since we have around 10,000 documents and 2,000 features for each, we have over 20,000,000 rows/features. This can be computationally expensive to process, so we will need to reduce the number of features by keeping the top most frequent words. Doing this carefully will not affect our analysis much because frequent words are able to capture the main stylistic and lexical differences between english language and translated text. However, we will need to be careful not to over-reduce because removing too many features may cause us to lose important discriminative patterns.  

The number of rows/features is still very large. The next option is to reduce the data (document) we have. (Note: usually we want to avoid this because the more data, the better. The model will see more example and learn patterns that help it classify unseen data.)

In [3]:
#Adjust as needed
num_data = 8000
top_words = 1000

#Keep top n (num_data) documents 
kept_eng_docs = original_eng['docid'].unique()[:num_data]
kept_trans_docs = translated['docid'].unique()[:num_data]

#For each document, keep top n (top_words) frequent words
reduced_eng_data = original_eng[original_eng['docid'].isin(kept_eng_docs)].groupby('docid').head(top_words)
reduced_trans_data = translated[translated['docid'].isin(kept_trans_docs)].groupby('docid').head(top_words)

#Save as new csv file
reduced_eng_data.to_csv("Reduced_Data/Reduced_original_doc.csv", index=False)
reduced_trans_data.to_csv("Reduced_Data/Reduced_translated_doc.csv", index=False)

Concatenate the reduced original (English) and translated files into one big csv file:

In [4]:
orig = pd.read_csv("Reduced_Data/Reduced_original_doc.csv")
trans = pd.read_csv("Reduced_Data/Reduced_translated_doc.csv")

#Add labels -> 0 = original & 1 = translated
orig['label'] = 0
trans['label'] = 1

#Combine datasets
data_combined = pd.concat([orig, trans], ignore_index=True)
data_combined.to_csv("Reduced_Data/Concatenated_data.csv", index=False)

### Multinomial Naive Bayes
**More Info:** https://www.geeksforgeeks.org/machine-learning/multinomial-naive-bayes/#:~:text=Multinomial%20Naive%20Bayes%20classifies%20text,as%20spam%20or%20not%20spam. 

In [5]:
#Make a safe copy to work with
data = data_combined.copy()
#Need to prepare data so that there is no bad rows where docid, term, or count is missing
data['count'] = pd.to_numeric(data['count'], errors='coerce')
data['term'] = data['term'].astype(str)
data = data.dropna(subset=['docid', 'term', 'count'])
data['count'] = data['count'].astype(int)

#Dictionary of docid with its terms and count
doc_dicts = data.groupby('docid').apply(lambda x: x.set_index('term')['count'].to_dict()).tolist()
#Label for each document
y = data.groupby('docid')['label'].first().values  

vectorizer = DictVectorizer(sparse=True)
X = vectorizer.fit_transform(doc_dicts)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = MultinomialNB()
model.fit(X_train, y_train)

#Get predictions and probabilities on held out data
y_pred = model.predict(X_test)
y_pred_probability = model.predict_proba(X_test)

#Model Performance Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision and Recall for each class:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8971875
Precision and Recall for each class:
              precision    recall  f1-score   support

           0       0.92      0.86      0.89      1588
           1       0.87      0.93      0.90      1612

    accuracy                           0.90      3200
   macro avg       0.90      0.90      0.90      3200
weighted avg       0.90      0.90      0.90      3200



**Analysis:** 

The model performs well with around 89.7% accuracy. This means that it is able to successfully classify many documents into its correct category. For class 0 (original text), the model has higher precision (0.92) than recall (0.86), indicating that it is usually correct when predicting class 0 but it misses true instances of that class. For class 1 (translated text), the pattern is reversed where precision (0.87) is lower than recall (0.93), indicating that it is able to successfully identify most true class 1 documents but makes slightly more false positives prediction. Overall, both classes have similar F1-scores, so performance is balanced, with a small tendency to predict class 1 more often. 

The performance is surprisingly good considering the model only relies on a simple bag-of-words (BOW) approach. This means that differences in word frequencies alone are enough to separate the two classes. Even without considering word order or meaning, the model can identify patterns based on how often words appear. This suggests that translated text tends to use certain words more or less consistently than original English text, which helps distinguish between the two.

In [24]:
print("Top-20 Most Probable Words for Each Category:")
feature_names = vectorizer.get_feature_names_out()
for i in range(len(model.classes_)):
    label = model.classes_[i]
    log_probs = model.feature_log_prob_[i]
    top20_indices = log_probs.argsort()[-20:][::-1]
    doc_type = "Original - English" if label == 0 else "Translated"
    print(f"\nCategory: {label} ({doc_type})")
    for idx in top20_indices:
        print(f"  - {feature_names[idx]} (log_prob: {log_probs[idx]:.4f})")


Top-20 Most Probable Words for Each Category:

Category: 0 (Original - English)
  - the (log_prob: -2.8171)
  - and (log_prob: -3.4388)
  - to (log_prob: -3.5570)
  - a (log_prob: -3.6652)
  - of (log_prob: -3.7076)
  - i (log_prob: -3.8548)
  - he (log_prob: -3.9694)
  - in (log_prob: -4.1003)
  - was (log_prob: -4.1428)
  - it (log_prob: -4.2176)
  - that (log_prob: -4.3465)
  - you (log_prob: -4.3525)
  - she (log_prob: -4.4776)
  - his (log_prob: -4.5039)
  - her (log_prob: -4.5760)
  - s (log_prob: -4.6177)
  - had (log_prob: -4.6618)
  - with (log_prob: -4.7858)
  - on (log_prob: -4.8338)
  - for (log_prob: -4.8392)

Category: 1 (Translated)
  - the (log_prob: -2.7435)
  - and (log_prob: -3.4469)
  - to (log_prob: -3.5113)
  - of (log_prob: -3.6383)
  - a (log_prob: -3.6797)
  - in (log_prob: -4.0107)
  - i (log_prob: -4.0155)
  - he (log_prob: -4.0225)
  - was (log_prob: -4.1760)
  - that (log_prob: -4.2881)
  - it (log_prob: -4.3475)
  - you (log_prob: -4.4249)
  - his (log_pro

**Analysis:**

Frequent words are needed to help capture stylistic differences in original and translated text, but in this case we see that the most frequent words are stop-words and they appear almost identical in both classes. Because of this, they probably do not help in distinguishing between the two classes. So the next best step is to remove these stop-words and focus on informative words that still appear often enough to show meaningful patterns, which should help the model better differentiate the two classes.

In [7]:
wrong_predictions = []
for i in range(len(y_test)):
    if y_pred[i] != y_test[i]:
        pred_confidence = np.max(y_pred_probability[i])
        wrong_predictions.append({
            'gold_label': y_test[i], 
            'pred_label': y_pred[i],
            'pred_confidence': pred_confidence
        })

wrong_predictions.sort(key=lambda x: x['pred_confidence'], reverse=True)
top_5_wrong = wrong_predictions[:5]

print("\nTop 5 wrong prediction with the highest confidence:\n")
for i, top_wrong in enumerate(top_5_wrong, 1):
    print(f"Prediction {i}:")
    print(f"   True Label: {top_wrong['gold_label']}")
    print(f"   Predicted Label: {top_wrong['pred_label']}")
    print(f"   Confidence: {top_wrong['pred_confidence']}")
    print()


Top 5 wrong prediction with the highest confidence:

Prediction 1:
   True Label: 1
   Predicted Label: 0
   Confidence: 1.0

Prediction 2:
   True Label: 0
   Predicted Label: 1
   Confidence: 1.0

Prediction 3:
   True Label: 1
   Predicted Label: 0
   Confidence: 1.0

Prediction 4:
   True Label: 0
   Predicted Label: 1
   Confidence: 1.0

Prediction 5:
   True Label: 0
   Predicted Label: 1
   Confidence: 1.0



**Analysis:**

We have seen that Naive Bayes performs very well, achieving around 89.7% accuracy in distinguishing between the two classes. However, the error analysis reveals a key issue. The model is often overconfident, assigning high probability (1.0) to the incorrect class. This occurs because Naive Bayes assumes all words are independent, but in reality words are often correlated. When correlated features all favor the incorrect class, they are treated as separate features, giving it higher confidence despite being incorrect. 

Therefore while the model’s predictions are fairly accurate overall, its probability estimates are unreliable. To improve this, we could use models like logistic regression to handle multiple feature relationships. Also, n-grams can help capture word dependencies beyond simple bag-of-words. (Note: Since the data is already represented as BOW, word order information is lost, so n-gram features is inapplicable.)

### Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression

#Reuse the same X_train, X_test, y_train, y_test prepared for Naive Bayes
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

#Get predictions and probabilities on held out data
lr_y_pred = lr_model.predict(X_test)
lr_y_pred_probability = lr_model.predict_proba(X_test)

#Model Performance Evaluation
print("Accuracy:", accuracy_score(y_test, lr_y_pred))
print("Precision and Recall for each class:")
print(classification_report(y_test, lr_y_pred))

Accuracy: 0.9634375
Precision and Recall for each class:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      1588
           1       0.96      0.96      0.96      1612

    accuracy                           0.96      3200
   macro avg       0.96      0.96      0.96      3200
weighted avg       0.96      0.96      0.96      3200



**Analysis:**

Logistic regression achieves 96.9% accuracy, which is a significant jump over Naive Bayes at 89.7%. This is a much larger improvement than you might expect from just swapping classifiers — it suggests that the independence assumption in Naive Bayes was genuinely hurting performance. Logistic regression learns a weighted combination of all features jointly, so it can handle the correlations between words more effectively instead of treating every word as if it contributes independently.

Looking at the per-class numbers, both classes come out at precision 0.97 and recall 0.97, with identical F1-scores of 0.97. The model is almost perfectly balanced — there is no meaningful tendency to favor one class over the other. This is a clean result and shows that logistic regression is picking up on a consistent signal across both original and translated text rather than leaning on class imbalances or frequency biases.

In [9]:
#Top-20 highest weighted words for each category
print("Top-20 Highest Weighted Words for Each Category:")
feature_names = vectorizer.get_feature_names_out()

#For binary classification, coef_ has shape (1, n_features)
#Positive coefficients push toward class 1 (translated), negative toward class 0 (original)
coefficients = lr_model.coef_[0]

top_original_indices = coefficients.argsort()[:20]
print("\nCategory: 0 (Original - English)")
for idx in top_original_indices:
    print(f"  - {feature_names[idx]} (coef: {coefficients[idx]:.4f})")

top_translated_indices = coefficients.argsort()[-20:][::-1]
print("\nCategory: 1 (Translated)")
for idx in top_translated_indices:
    print(f"  - {feature_names[idx]} (coef: {coefficients[idx]:.4f})")

Top-20 Highest Weighted Words for Each Category:

Category: 0 (Original - English)
  - across (coef: -0.1987)
  - except (coef: -0.1931)
  - week (coef: -0.1690)
  - suppose (coef: -0.1614)
  - expected (coef: -0.1507)
  - done (coef: -0.1467)
  - zebra (coef: -0.1435)
  - paused (coef: -0.1400)
  - wondered (coef: -0.1340)
  - reb (coef: -0.1335)
  - many (coef: -0.1331)
  - minutes (coef: -0.1319)
  - ice (coef: -0.1316)
  - everybody (coef: -0.1306)
  - itself (coef: -0.1292)
  - back (coef: -0.1289)
  - stephen (coef: -0.1274)
  - fire (coef: -0.1264)
  - hot (coef: -0.1231)
  - interest (coef: -0.1230)

Category: 1 (Translated)
  - longer (coef: 0.2065)
  - without (coef: 0.2025)
  - everything (coef: 0.1962)
  - belacqua (coef: 0.1897)
  - understand (coef: 0.1859)
  - completely (coef: 0.1733)
  - pat (coef: 0.1600)
  - able (coef: 0.1578)
  - same (coef: 0.1489)
  - whether (coef: 0.1480)
  - set (coef: 0.1467)
  - tears (coef: 0.1424)
  - managed (coef: 0.1365)
  - eyes (coef:

**Analysis:**

Unlike Naive Bayes, where the top features were almost entirely stop-words (the, and, to, of, ...), logistic regression surfaces actual content words with meaningful coefficients. Words like "across", "except", "suppose", "wondered", and "paused" point toward original English, while "longer", "without", "everything", "understand", and "completely" point toward translated text. This is a much more interpretable result.

This difference makes sense given how the two models work. Naive Bayes ranks features by raw log-probability, so the most frequent words always dominate. Logistic regression instead learns coefficients that reflect how much each word shifts the decision boundary, which means very common words that appear equally in both classes get low weights and less distinctive words can rise to the top.

The translated-text side is particularly interesting — words like "longer", "without", "completely", and "managed" are fairly neutral in English but may reflect common patterns in how certain source languages phrase things when translated. Some words like "belacqua", "isabeau", and "maigret" are proper nouns from specific translated works, which suggests the model may be partially picking up on document-level signals rather than pure stylistic patterns. That is worth keeping in mind when interpreting these results.

In [10]:
lr_wrong_predictions = []
for i in range(len(y_test)):
    if lr_y_pred[i] != y_test[i]:
        pred_confidence = np.max(lr_y_pred_probability[i])
        lr_wrong_predictions.append({
            'gold_label': y_test[i],
            'pred_label': lr_y_pred[i],
            'pred_confidence': pred_confidence
        })

lr_wrong_predictions.sort(key=lambda x: x['pred_confidence'], reverse=True)
lr_top_5_wrong = lr_wrong_predictions[:5]

print("\nTop 5 wrong predictions with the highest confidence:\n")
for i, top_wrong in enumerate(lr_top_5_wrong, 1):
    print(f"Prediction {i}:")
    print(f"   True Label: {top_wrong['gold_label']}")
    print(f"   Predicted Label: {top_wrong['pred_label']}")
    print(f"   Confidence: {top_wrong['pred_confidence']}")
    print()


Top 5 wrong predictions with the highest confidence:

Prediction 1:
   True Label: 1
   Predicted Label: 0
   Confidence: 0.9999999999999019

Prediction 2:
   True Label: 0
   Predicted Label: 1
   Confidence: 0.9999999998721159

Prediction 3:
   True Label: 1
   Predicted Label: 0
   Confidence: 0.9999999955712421

Prediction 4:
   True Label: 1
   Predicted Label: 0
   Confidence: 0.9999999874674006

Prediction 5:
   True Label: 0
   Predicted Label: 1
   Confidence: 0.9999999483449136



**Analysis:**

The overconfidence problem from Naive Bayes has not gone away. All five wrong predictions still have confidence values that are essentially 1.0 (e.g., 0.9999999999999072), meaning logistic regression is just as extreme in its certainty when it gets things wrong. This is a bit surprising since logistic regression is often expected to be better calibrated than Naive Bayes, but with this many features it can still push probabilities toward the extremes when the evidence consistently lines up on one side.

What is notable here is that even with 96.9% accuracy — meaning far fewer errors overall — the mistakes that do happen are still high-confidence ones. The model is not unsure on the cases it gets wrong; it is just confidently incorrect. These are likely genuinely hard cases where the document's word distribution closely matches the wrong class, possibly a translation that reads unusually naturally in English or an original text that happens to use vocabulary patterns more typical of translated prose.

Overall, logistic regression is a clear improvement over Naive Bayes in terms of raw accuracy (96.9% vs. 89.7%), but both models share the same weakness when it comes to probability calibration on their errors.

### Multinomial Naive Bayes w/ Stopwords Removal

In [6]:
#Find indices of rows where term is a stop word & drop the rows
stopword_idx = data[data['term'].str.lower().isin(ENGLISH_STOP_WORDS)].index
data = data.drop(stopword_idx)

doc_dicts = data.groupby('docid').apply(lambda x: x.set_index('term')['count'].to_dict()).tolist()
y = data.groupby('docid')['label'].first().values  

vectorizer = DictVectorizer(sparse=True)
X = vectorizer.fit_transform(doc_dicts)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_probability = model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision and Recall for each class:")
print(classification_report(y_test, y_pred))


Accuracy: 0.9084375
Precision and Recall for each class:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      1588
           1       0.89      0.94      0.91      1612

    accuracy                           0.91      3200
   macro avg       0.91      0.91      0.91      3200
weighted avg       0.91      0.91      0.91      3200



In [7]:
print("Top-20 Most Probable Words for Each Category:")
feature_names = vectorizer.get_feature_names_out()
for i in range(len(model.classes_)):
    label = model.classes_[i]
    log_probs = model.feature_log_prob_[i]
    top20_indices = log_probs.argsort()[-20:][::-1]
    doc_type = "Original - English" if label == 0 else "Translated"
    print(f"\nCategory: {label} ({doc_type})")
    for idx in top20_indices:
        print(f"  - {feature_names[idx]} (log_prob: {log_probs[idx]:.4f})")

Top-20 Most Probable Words for Each Category:

Category: 0 (Original - English)
  - s (log_prob: -3.6413)
  - t (log_prob: -3.9393)
  - said (log_prob: -4.0640)
  - like (log_prob: -4.7460)
  - know (log_prob: -5.1206)
  - time (log_prob: -5.1832)
  - don (log_prob: -5.2276)
  - just (log_prob: -5.2393)
  - man (log_prob: -5.2583)
  - did (log_prob: -5.2591)
  - d (log_prob: -5.2597)
  - m (log_prob: -5.5019)
  - little (log_prob: -5.5058)
  - way (log_prob: -5.5164)
  - ll (log_prob: -5.5786)
  - think (log_prob: -5.5959)
  - got (log_prob: -5.5981)
  - come (log_prob: -5.5995)
  - thought (log_prob: -5.6105)
  - good (log_prob: -5.6436)

Category: 1 (Translated)
  - s (log_prob: -3.7109)
  - t (log_prob: -4.1080)
  - said (log_prob: -4.4946)
  - like (log_prob: -4.7942)
  - time (log_prob: -5.0646)
  - did (log_prob: -5.1992)
  - man (log_prob: -5.2064)
  - don (log_prob: -5.3247)
  - know (log_prob: -5.3484)
  - just (log_prob: -5.3917)
  - come (log_prob: -5.4554)
  - little (log_p

**Analysis:**

Removing stopwords slightly improved the Multinomial NB classifier, increasing the accuracy from 89.7% to 90.8% and slightly boosting the precision and recall for each class. Without stopword removal, the model focused mostly on common words like "the", "and", and "of" which appears in both original and translated text and don't help distinguish between them. But after removing these stopwords, it reveals some differences between the class. For example, words like "did", "thought", and "good" appear more in original (English) than in translated text, suggesting that the model is picking up on subtly stylistic patterns. However, many top words still overlap between the both categories which explains why the accuracy improvement was modest rather than dramatic.


### Logistic Regression w/ Stopwords Removal

In [13]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

lr_y_pred = lr_model.predict(X_test)
lr_y_pred_probability = lr_model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, lr_y_pred))
print("Precision and Recall for each class:")
print(classification_report(y_test, lr_y_pred))

Accuracy: 0.9634375
Precision and Recall for each class:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      1588
           1       0.96      0.96      0.96      1612

    accuracy                           0.96      3200
   macro avg       0.96      0.96      0.96      3200
weighted avg       0.96      0.96      0.96      3200



In [14]:
#Top-20 highest weighted words for each category
print("Top-20 Highest Weighted Words for Each Category:")
feature_names = vectorizer.get_feature_names_out()

#For binary classification, coef_ has shape (1, n_features)
#Positive coefficients push toward class 1 (translated), negative toward class 0 (original)
coefficients = lr_model.coef_[0]

top_original_indices = coefficients.argsort()[:20]
print("\nCategory: 0 (Original - English)")
for idx in top_original_indices:
    print(f"  - {feature_names[idx]} (coef: {coefficients[idx]:.4f})")

top_translated_indices = coefficients.argsort()[-20:][::-1]
print("\nCategory: 1 (Translated)")
for idx in top_translated_indices:
    print(f"  - {feature_names[idx]} (coef: {coefficients[idx]:.4f})")

Top-20 Highest Weighted Words for Each Category:

Category: 0 (Original - English)
  - suppose (coef: -0.2713)
  - wondered (coef: -0.2305)
  - expected (coef: -0.2188)
  - paused (coef: -0.2157)
  - week (coef: -0.1915)
  - trying (coef: -0.1878)
  - stephen (coef: -0.1769)
  - early (coef: -0.1676)
  - wish (coef: -0.1661)
  - guess (coef: -0.1624)
  - waited (coef: -0.1619)
  - watched (coef: -0.1610)
  - held (coef: -0.1590)
  - minutes (coef: -0.1560)
  - ain (coef: -0.1548)
  - miles (coef: -0.1527)
  - watching (coef: -0.1524)
  - mind (coef: -0.1520)
  - hot (coef: -0.1519)
  - afternoon (coef: -0.1513)

Category: 1 (Translated)
  - longer (coef: 0.2546)
  - today (coef: 0.2162)
  - completely (coef: 0.2128)
  - understand (coef: 0.2076)
  - evening (coef: 0.1932)
  - able (coef: 0.1886)
  - stuck (coef: 0.1846)
  - calm (coef: 0.1829)
  - suddenly (coef: 0.1684)
  - single (coef: 0.1673)
  - happened (coef: 0.1642)
  - order (coef: 0.1594)
  - word (coef: 0.1579)
  - clearly (

**Analysis**:

Logistic Regression performs much better than NB, reaching about 96.3% accuracy in both cases, with very balanced precision and recall (about 0.96 for both classes). Unlike NB, removing stopwords has no effect on performance, which suggests that Logistic Regression is better at down-weighing uninformative features like common words on its own. Additionally, looking at the top weighted words, Logistic Regression captures more informative features that help separate the two classes. For original English, words like “watching,” “guess,” and “wondered” suggest a more natural and conversational writing style. For translated text, words like “completely,” “understand,” and “managed” suggest a more formal style.